In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import re
import os

# Define file paths (adjust if your BGL.log is stored elsewhere)
log_file_path = '../data/raw/BGL.log'

def parse_bgl_log(file_path, max_rows=100000):
    """
    Parses the BGL log file.
    Extracts the label (normal/anomaly) and the core log message.
    """
    parsed_data = []
    
    if not os.path.exists(file_path):
        print(f"Error: Could not find {file_path}. Please check the path.")
        return None
        
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= max_rows: # Limit rows for initial testing and memory management
                break
                
            parts = line.strip().split()
            if len(parts) < 10:
                continue
                
            # BGL format: Label is the first token. '-' means normal.
            label_str = parts[0]
            label = 0 if label_str == '-' else 1
            
            # The core message usually starts after the standard system info headers
            # Joining the rest of the line to form the sequence for the LSTM
            message = " ".join(parts[9:]) 
            
            parsed_data.append({'label': label, 'message': message})
            
    df = pd.DataFrame(parsed_data)
    print(f"Successfully parsed {len(df)} log entries.")
    print(f"Normal logs: {len(df[df['label'] == 0])} | Anomaly logs: {len(df[df['label'] == 1])}")
    return df

# Run the parser
bgl_df = parse_bgl_log(log_file_path, max_rows=100000)
if bgl_df is not None:
    display(bgl_df.head())

Successfully parsed 100000 log entries.
Normal logs: 97391 | Anomaly logs: 2609


,label,message
0,0,instruction cache parity error corrected
1,0,instruction cache parity error corrected
2,0,instruction cache parity error corrected
3,0,instruction cache parity error corrected
4,0,instruction cache parity error corrected


In [3]:
import re
from collections import Counter

# 1. Clean the log messages (remove hex, IPs, and special characters)
def clean_log(text):
    text = str(text).lower()
    text = re.sub(r'0x[a-f0-9]+', 'HEX', text) # Replace hex codes with a generic 'HEX' token
    text = re.sub(r'\d+\.\d+\.\d+\.\d+', 'IP', text) # Replace IP addresses
    text = re.sub(r'\d+', 'NUM', text) # Replace standalone numbers
    text = re.sub(r'[^\w\s]', ' ', text) # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

print("Cleaning log messages...")
bgl_df['cleaned_message'] = bgl_df['message'].apply(clean_log)

# 2. Build the Vocabulary
print("Building vocabulary...")
all_words = ' '.join(bgl_df['cleaned_message']).split()
word_counts = Counter(all_words)

# Only keep words that appear more than once to reduce noise
vocab = {word for word, count in word_counts.items() if count > 1}
word2idx = {word: idx + 2 for idx, word in enumerate(vocab)}
word2idx['<PAD>'] = 0  # Padding token
word2idx['<UNK>'] = 1  # Unknown word token

print(f"Vocabulary size: {len(word2idx)} words")

# 3. Tokenize and Pad Sequences
MAX_LEN = 30 # Adjust based on your average log length

def encode_and_pad(text):
    tokens = text.split()
    # Convert to indices, use <UNK> if word not in vocab
    seq = [word2idx.get(word, 1) for word in tokens]
    # Pad or truncate
    if len(seq) < MAX_LEN:
        seq = seq + [0] * (MAX_LEN - len(seq))
    else:
        seq = seq[:MAX_LEN]
    return seq

print("Encoding and padding sequences...")
X = np.array(bgl_df['cleaned_message'].apply(encode_and_pad).tolist())
y = bgl_df['label'].values

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")

Cleaning log messages...
Building vocabulary...
Vocabulary size: 186 words
Encoding and padding sequences...
Feature matrix X shape: (100000, 30)
Target vector y shape: (100000,)


In [4]:
# 1. Train/Test Split
# We use stratify=y to ensure both sets have the same ratio of normal to anomalous logs
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define PyTorch Dataset
class LogDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X_data = torch.tensor(X_data, dtype=torch.long)
        self.y_data = torch.tensor(y_data, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X_data)
        
    def __getitem__(self, idx):
        return self.X_data[idx], self.y_data[idx]

# Create Datasets and DataLoaders
train_dataset = LogDataset(X_train, y_train)
test_dataset = LogDataset(X_test, y_test)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Testing batches: {len(test_loader)}")

# 3. Define the LSTM Model Architecture
class LogAnomalyLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super(LogAnomalyLSTM, self).__init__()
        # Embedding layer turns token integers into dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # LSTM processes the sequence and captures sequential dependencies (e.g., event A followed by event B)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        # Fully connected layer for the final classification
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # We only need the LSTM's final hidden state to make our classification decision
        final_hidden = hidden[-1, :, :]
        out = self.fc(final_hidden)
        
        # Sigmoid squashes the output between 0 and 1 (0 = Safe, 1 = Anomaly)
        return self.sigmoid(out).squeeze()

# Initialize the model
vocab_size = len(word2idx)
embed_dim = 32
hidden_dim = 64
output_dim = 1

model = LogAnomalyLSTM(vocab_size, embed_dim, hidden_dim, output_dim)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"\nModel initialized and moved to: {device}")
print(model)

Training batches: 625
Testing batches: 157

Model initialized and moved to: cuda
LogAnomalyLSTM(
  (embedding): Embedding(186, 32, padding_idx=0)
  (lstm): LSTM(32, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [5]:
import torch.optim as optim
import json
import os

# 1. Define Loss Function and Optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy is standard for 0/1 classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Training Loop
epochs = 5
print("Starting Training...")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        # Move data to GPU
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        # Calculate training accuracy
        predicted = (outputs >= 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_acc = 100 * correct_train / total_train
    
    # 3. Validation Loop (Test Set)
    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            
            # Calculate testing accuracy
            predicted = (outputs >= 0.5).float()
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()
            
    test_acc = 100 * correct_test / total_test
    
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss/len(test_loader):.4f} | Test Acc: {test_acc:.2f}%")

# 4. Save the Model Weights and Vocabulary
os.makedirs('../data/processed', exist_ok=True)

torch.save(model.state_dict(), 'log_lstm_best.pth')
print("\nModel weights saved to 'log_lstm_best.pth'")

with open('../data/processed/log_word2idx.json', 'w') as f:
    json.dump(word2idx, f)
print("Vocabulary saved to '../data/processed/log_word2idx.json'")

Starting Training...
Epoch 1/5 | Train Loss: 0.0741 | Train Acc: 98.76% | Test Loss: 0.0062 | Test Acc: 99.93%
Epoch 2/5 | Train Loss: 0.0014 | Train Acc: 99.99% | Test Loss: 0.0002 | Test Acc: 100.00%
Epoch 3/5 | Train Loss: 0.0003 | Train Acc: 100.00% | Test Loss: 0.0001 | Test Acc: 100.00%
Epoch 4/5 | Train Loss: 0.0004 | Train Acc: 100.00% | Test Loss: 0.0001 | Test Acc: 100.00%
Epoch 5/5 | Train Loss: 0.0003 | Train Acc: 100.00% | Test Loss: 0.0000 | Test Acc: 100.00%

Model weights saved to 'log_lstm_best.pth'
Vocabulary saved to '../data/processed/log_word2idx.json'


In [6]:
def predict_log_anomaly(log_text, model, device):
    """
    Cleans, encodes, and predicts whether a single log string is normal or anomalous.
    """
    # 1. Clean and encode the raw log string
    cleaned_text = clean_log(log_text)
    encoded_seq = encode_and_pad(cleaned_text)
    
    # 2. Convert to PyTorch tensor and add a batch dimension (shape: 1 x MAX_LEN)
    tensor_input = torch.tensor(encoded_seq, dtype=torch.long).unsqueeze(0).to(device)
    
    # 3. Run inference
    model.eval()
    with torch.no_grad():
        probability = model(tensor_input).item()
        
    # 4. Output results
    verdict = "🚨 ANOMALY DETECTED" if probability >= 0.5 else "✅ NORMAL KERNEL EVENT"
    
    print(f"Raw Log: {log_text}")
    print(f"Cleaned: {cleaned_text}")
    print(f"Anomaly Probability: {probability * 100:.2f}%")
    print(f"Verdict: {verdict}")
    print("-" * 50)

# Test 1: A standard operational log (Normal)
normal_log = "RAS KERNEL INFO instruction cache parity error corrected"
predict_log_anomaly(normal_log, model, device)

# Test 2: A critical system failure log (Anomaly)
anomaly_log = "RAS KERNEL FATAL data TLB error interrupt"
predict_log_anomaly(anomaly_log, model, device)


Raw Log: RAS KERNEL INFO instruction cache parity error corrected
Cleaned: ras kernel info instruction cache parity error corrected
Anomaly Probability: 0.00%
Verdict: ✅ NORMAL KERNEL EVENT
--------------------------------------------------
Raw Log: RAS KERNEL FATAL data TLB error interrupt
Cleaned: ras kernel fatal data tlb error interrupt
Anomaly Probability: 99.93%
Verdict: 🚨 ANOMALY DETECTED
--------------------------------------------------
